# 02 · Prompting progresivo: de un prompt suelto a conocimiento integrado

**Caso:** clasificar tickets de soporte de telecomunicaciones en `billing`, `technical`, `cancellation` u `other`. El formato de salida siempre será JSON para poder evaluar.

La ruta: empezamos con un prompt sin estructura, vemos por qué falla, le damos estructura y después aplicamos técnicas encima de esa base.

## 0. Preparación

Misma configuración que el notebook 01: llaves en `.env`, LM Studio activo.

In [13]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from receipt_validation.config import load_settings

settings = load_settings(ROOT)
print(f"Project root: {ROOT}")

Project root: /Users/andrei/Documents/Projects/Curso LLM Codes/session_2_architecture


Definimos el caso y dos helpers: `call_model` hace la llamada al modelo local y `show` presenta resultados legibles.

In [14]:
import requests
from IPython.display import Markdown, display

LOCAL_URL = f"{settings['lmstudio_base_url'].rstrip('/')}/chat/completions"
LOCAL_MODEL = settings["default_lmstudio_model"]

TICKET = "Desde ayer no tengo señal. Reinicié el módem dos veces y la luz LOS sigue roja."
CATEGORIES = ["billing", "technical", "cancellation", "other"]

def call_model(prompt, temperature=0.0, max_tokens=1024):
    payload = {
        "model": LOCAL_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    response = requests.post(LOCAL_URL, json=payload, timeout=120)
    response.raise_for_status()
    message = response.json()["choices"][0]["message"]
    return message.get("content") or message.get("reasoning_content") or ""

def show(title, body):
    display(Markdown(f"#### {title}\n\n{body}\n\n---"))

## 1. Prompt sin estructura

Primero lo que haría cualquiera: pedirle al modelo que clasifique, sin más. Observen la salida: formato libre, categorías inventadas, imposible de evaluar en automático.

In [15]:
naive_prompt = f"What kind of problem is this? {TICKET}"
show("Prompt", naive_prompt)
show("Result", call_model(naive_prompt))

#### Prompt

What kind of problem is this? Desde ayer no tengo señal. Reinicié el módem dos veces y la luz LOS sigue roja.

---

KeyboardInterrupt: 

## 2. El mismo prompt, con estructura

Cinco piezas: rol, categorías cerradas, regla de ambigüedad, entrada delimitada y contrato de salida en JSON. La tarea no cambió; cambió que ahora la salida es predecible y evaluable.

In [ ]:
structured_prompt = f"""You classify telecom support tickets.
Choose exactly one category from: {CATEGORIES}.
If the request is ambiguous, choose other.

<ticket>{TICKET}</ticket>

Return only JSON: {{"category": "...", "confidence": 0.0, "reason": "one short sentence"}}
"""
show("Prompt", structured_prompt)
show("Result", call_model(structured_prompt))

#### Prompt

You classify telecom support tickets.
Choose exactly one category from: ['billing', 'technical', 'cancellation', 'other'].
If the request is ambiguous, choose other.

<ticket>Desde ayer no tengo señal. Reinicié el módem dos veces y la luz LOS sigue roja.</ticket>

Return only JSON: {"category": "...", "confidence": 0.0, "reason": "one short sentence"}


---

#### Result

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The ticket describes a loss of signal and troubleshooting steps for the modem."
}
```

---

## 3. Estabilidad: ambos prompts a distintas temperaturas

Corremos las dos versiones a tres temperaturas. El prompt sin estructura se degrada al subir la temperatura; el estructurado mantiene el formato incluso con alta diversidad. La estructura es lo que hace al sistema robusto, no la temperatura baja.

In [ ]:
prompts = {"naive": naive_prompt, "structured": structured_prompt}

for name, prompt in prompts.items():
    for temperature in (0.0, 0.7, 1.2):
        result = call_model(prompt, temperature=temperature)
        show(f"{name} · temperature {temperature}", result)

#### naive · temperature 0.0

Este es un **problema de conectividad o fallo en la línea física** (Loss of Signal).

El hecho de que la luz **LOS (Loss of Signal)** siga roja indica que el módem no está logrando establecer una conexión estable con la red de tu proveedor de servicios de internet (ISP).

Aquí te explico qué significa y qué puedes hacer para intentar solucionarlo:

### 💡 ¿Qué significa la luz LOS roja?

La luz

---

#### naive · temperature 0.7

Este es un **problema de conectividad técnica** o una **falla en el servicio de internet/telecomunicación**.

El mensaje que describes es muy específico y ya has realizado el primer paso de diagnóstico (reiniciar el módem), lo cual es correcto.

Aquí te explico qué significa y cuáles son los pasos a seguir:

### 💡 ¿Qué significa la luz LOS roja?

La luz **LOS (Loss of Signal)** roja indica que el módem no está recibiendo una señal adecuada del proveedor de internet. Esto generalmente apunta a uno de los siguientes problemas:

1. **Falla en la línea externa:** Hay un corte o una mala

---

#### naive · temperature 1.2

Este es un **problema de conectividad

---

#### structured · temperature 0.0

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The ticket describes a loss of signal and troubleshooting steps for the modem."
}
```

---

#### structured · temperature 0.7

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The ticket describes a loss of signal and troubleshooting related to the modem."
}
```

---

#### structured · temperature 1.2

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The ticket describes a connectivity issue with the modem and loss of signal."
}
```

---

## 4. One-shot

Sobre la base estructurada añadimos un ejemplo representativo. El ejemplo enseña a la vez la decisión y el formato; conviene que sea correcto, breve y cercano al borde que queremos aclarar.

In [ ]:
one_shot_prompt = f"""You classify telecom support tickets.
Categories: {CATEGORIES}. Return only JSON.

Example:
<ticket>No llegó mi factura de este mes.</ticket>
{{"category": "billing", "confidence": 0.98, "reason": "The request concerns an invoice."}}

Now classify:
<ticket>{TICKET}</ticket>
"""
show("Result", call_model(one_shot_prompt))

#### Result

```json
{
  "category": "technical",
  "confidence": 0.98,
  "reason": "The ticket describes a loss of signal and issues with the modem equipment (LOS light remains red), which is a technical fault."
}
```

---

## 5. Few-shot

Varios ejemplos cubren clases y casos límite. No se trata de llenar contexto: cada ejemplo debe aportar una regla observable. Ojo con el orden y con sobrerrepresentar una clase.

In [ ]:
examples = [
    ("Me cobraron dos veces el mismo mes.", "billing"),
    ("La luz LOS está roja y no navego.", "technical"),
    ("Quiero dar de baja el servicio hoy.", "cancellation"),
    ("¿Abren el domingo?", "other"),
]
formatted_examples = "\n".join(
    f'<ticket>{text}</ticket>\n{{"category": "{label}"}}' for text, label in examples
)

few_shot_prompt = f"""You classify telecom support tickets into {CATEGORIES}.
Learn the decision boundary from these examples:
{formatted_examples}

Classify <ticket>{TICKET}</ticket> and return only JSON with category, confidence, and reason.
"""
show("Result", call_model(few_shot_prompt))

#### Result

```json
{
  "category": "technical",
  "confidence": 0.95,
  "reason": "The ticket describes a loss of signal and persistent error lights (LOS), which clearly indicates a technical fault."
}
```

---

## 6. Chain-of-Thought (CoT)

La técnica clásica pide razonar paso a paso antes de responder. En producción suele ser mejor pedir una **justificación breve y verificable** con criterios observables: síntoma, evidencia y decisión.

In [ ]:
cot_prompt = f"""Classify the ticket into {CATEGORIES}.
Evaluate these observable criteria in order:
1. Identify the customer's explicit symptom.
2. Identify evidence tied to one category.
3. Resolve ambiguity using the closed category list.

<ticket>{TICKET}</ticket>

Return only JSON with category, confidence, and a concise evidence-based justification.
"""
show("Result", call_model(cot_prompt))

#### Result

```json
{
  "category": "technical",
  "confidence": 1.0,
  "justification": "The customer explicitly reports a loss of signal ('no tengo señal') and details troubleshooting steps (restarting the modem) along with a specific hardware indicator (LOS light is red), which clearly indicates a technical malfunction or connectivity issue."
}
```

---

## 7. Knowledge Generation & Knowledge Integration

Dos llamadas: primero generamos conocimiento del dominio, después lo integramos como contexto controlado para decidir. Esto permite inspeccionar, filtrar o sustituir el conocimiento antes de clasificar.

**Antecesor:** *Generated Knowledge Prompting* generaba conocimiento y lo añadía al prompt final. Separar las dos etapas mejora el control.

In [ ]:
knowledge_prompt = f"""Generate up to four short telecom support rules useful for classifying
this ticket. Include only domain knowledge; do not choose the final category.
<ticket>{TICKET}</ticket>"""

generated_knowledge = call_model(knowledge_prompt)
show("Generated knowledge", generated_knowledge)

#### Generated knowledge

*   **Rule 1:** Ausencia de señal persistente. (No signal persistence).
*   **Rule 2:** Indicador LOS en estado de error (rojo). (LOS indicator in error state (red)).
*   **Rule 3:** Fallo de conectividad tras reinicio. (Connectivity failure after restart).
*   **Rule 4:** Diagnóstico de hardware/señal fallido. (Faulty hardware/signal diagnosis).

---

Segunda etapa: inyectamos el conocimiento revisado y pedimos evidencia más un campo `used_knowledge` para auditar qué se usó.

In [ ]:
integration_prompt = f"""Classify the ticket into {CATEGORIES}.
Use the supplied knowledge only when it is supported by the ticket.
<knowledge>{generated_knowledge}</knowledge>
<ticket>{TICKET}</ticket>
Return only JSON with category, confidence, evidence, and used_knowledge.
"""
show("Result", call_model(integration_prompt))

#### Result

```json
{
  "category": "technical",
  "confidence": 1.0,
  "evidence": "Desde ayer no tengo señal. Reinicié el módem dos veces y la luz LOS sigue roja.",
  "used_knowledge": [
    "Rule 1: Ausencia de señal persistente.",
    "Rule 2: Indicador LOS en estado de error (rojo).",
    "Rule 3: Fall

---

## 8. Salida estructurada garantizada (Pydantic + JSON Schema)

Hasta ahora el JSON se pedía «por favor» dentro del prompt: el modelo puede romper el contrato en cualquier momento. La forma robusta es declararlo con un modelo de Pydantic y pasarlo como `response_json_schema`: el proveedor fuerza la salida a cumplir el esquema y la respuesta se valida de regreso a un objeto tipado. Lo aplicamos sobre nuestra mejor técnica: el prompt few-shot.

In [ ]:
from typing import Literal
from google import genai
from pydantic import BaseModel, Field

client = genai.Client(api_key=settings["google_api_key"])

class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "cancellation", "other"] = Field(description="One of the closed categories")
    confidence: float = Field(description="From 0.0 to 1.0")
    reason: str = Field(description="One short evidence-based sentence")

In [ ]:
response = client.models.generate_content(
    model=settings["default_gemini_model"],
    contents=[few_shot_prompt],
    config={
        "response_mime_type": "application/json",
        "response_json_schema": TicketClassification.model_json_schema(),
    },
)

result = TicketClassification.model_validate_json(response.text)
show("Structured result", f"```json\n{result.model_dump_json(indent=2)}\n```")
result

#### Structured result

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The customer describes a loss of signal and hardware indicators consistent with a service outage."
}
```

---

TicketClassification(category='technical', confidence=1.0, reason='The customer describes a loss of signal and hardware indicators consistent with a service outage.')

## 9. Cierre

- Zero-shot estructurado es la línea base; suele bastar para tareas claras.
- One/few-shot ayudan cuando la frontera entre clases necesita ejemplos.
- CoT o criterios estructurados sirven cuando la decisión requiere composición.
- Knowledge Generation & Integration sirve cuando hace falta conocimiento inspeccionable.
- Una técnica más compleja solo se conserva si mejora una métrica relevante.
- Cuando el formato es crítico no se pide: se garantiza con `response_json_schema`.

<details><summary><strong>Pausa y conclusión</strong></summary>

1. ¿Qué cambió entre el prompt suelto y el estructurado?
2. ¿Cuál de los dos resistió mejor la temperatura alta y por qué?
3. ¿Cuándo justificarían pagar dos llamadas (knowledge + integration)?
</details>